Hands-On Lab: Sprint 3 Kickoff & Text Cleaning      
● Step 1: Complete Sprint 3 planning and select the backlog tasks for integration and evaluation.       
● Step 2: Take a raw text sample (or the project's text data) and tokenize it.      
● Step 3: Apply a full cleaning pipeline: lowercase, remove punctuation, remove stop words, lemmatize.      
● Step 4: Verify the cleaning preserves task-critical words (e.g. negations for sentiment) and document the choices in Markdown.        

#### Load Dataset:

In [6]:
import pandas as pd


Reusing the same dataset as Sprint 2\day4 imdb reviews data !  

In [2]:
path = r"..\Data\IMDB Dataset of 50K Movie Reviews\IMDB Dataset.csv"

df = pd.read_csv(path) # header=None because this dataset does not use a normal header row

print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
print(df.isnull().sum())
print("")
print("Duplicates:", df.duplicated().sum())

review       0
sentiment    0
dtype: int64

Duplicates: 418


In [4]:
df = df.drop_duplicates().reset_index(drop=True)

print("New shape:", df.shape)
print("Duplicates left:", df.duplicated().sum())

New shape: (49582, 2)
Duplicates left: 0


### TEXT CLEANING:

In [ ]:
import nltk

import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

We are using:

word_tokenize → split sentences into tokens.    

stopwords → remove common low-information words.     

WordNetLemmatizer → reduce words to their base form.    

string.punctuation → help remove punctuation.        

In [8]:
# These are extra language resources NLTK needs.
# (tokenizer data) (downloaded once)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to C:\Users\Masters
[nltk_data]     Computer\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to C:\Users\Masters
[nltk_data]     Computer\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to C:\Users\Masters
[nltk_data]     Computer\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to C:\Users\Masters
[nltk_data]     Computer\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to C:\Users\Masters
[nltk_data]     Computer\AppData\Roaming\nltk_data...


True

#### `Preprocessing a Single review (first row)`:

##### Tokenization

In [10]:
# First, trying with a sample. 
# first review in the dataset is a positive review, so we will use that as our sample.

sample_review = df["review"].iloc[0] 

print(sample_review)

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

In [14]:
tokens = word_tokenize(sample_review) # turns the connected sentence into seperate words (each is a token).

print(tokens[:30])

['One', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'Oz', 'episode', 'you', "'ll", 'be', 'hooked', '.', 'They', 'are', 'right', ',', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with']


##### Lower-casing

In [15]:
tokens_lower = [token.lower()        # turns each token in 'tokens' to lower-case.
                for token in tokens]  # a loop so it iterates over each token.  

print (tokens_lower[:30])

['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'you', "'ll", 'be', 'hooked', '.', 'they', 'are', 'right', ',', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with']


##### Remove punctuation

In [20]:
tokens_no_punctuation = [token                               # take token 
                         for token in tokens_lower           # from lower-cased tokens
                         if token not in string.punctuation] # removes tokens like:  ? ! , . : ; 

                                                             # Simply,the function takes tokens if they are not not a punctuation sign. 
print(tokens_no_punctuation[:30])

['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'you', "'ll", 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me.', 'br']


##### Stop-word removal

In [17]:
stop_words = set( stopwords.words("English")) 

# These include common words such as: the a is and of not nor .
# but we don't blindly remove negation like "not" since it can affect the sentiment.

# so we do this :

stop_words.discard("not")
stop_words.discard("no")
stop_words.discard("nor")

In [21]:
tokens_no_stopwords = [token
                       for token in tokens_no_punctuation
                       if token not in stop_words] 
                                                    # Simply,the function takes tokens if they are not not a in stop_words set.  
                                                    
print(tokens_no_stopwords[:30])

['one', 'reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', "'ll", 'hooked', 'right', 'exactly', 'happened', 'me.', 'br', 'br', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show']


##### Lemmatization

In [ ]:
# Lemmatization tries to reduce related forms into a more consistent base form.
# cars -> car
# movies -> movie

lemmatizer = WordNetLemmatizer() # create the lemmatizer.

lemmatized_tokens = [ lemmatizer.lemmatize(token)
                     for token in tokens_no_stopwords]
                                                        # for each token in tokens_no_stopwords lemmatize the token.
                                                        
print (lemmatized_tokens[:30])

['one', 'reviewer', 'mentioned', 'watching', '1', 'oz', 'episode', "'ll", 'hooked', 'right', 'exactly', 'happened', 'me.', 'br', 'br', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scene', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show']


##### Joining the tokens back into a cleaned text (to view it):

In [23]:
cleaned_review = " ".join(lemmatized_tokens)

print (cleaned_review)

one reviewer mentioned watching 1 oz episode 'll hooked right exactly happened me. br br first thing struck oz brutality unflinching scene violence set right word go trust not show faint hearted timid show pull no punch regard drug sex violence hardcore classic use word. br br called oz nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inwards privacy not high agenda em city home many .. aryan muslim gangsta latino christian italian irish .... scuffle death stare dodgy dealing shady agreement never far away. br br would say main appeal show due fact go show would n't dare forget pretty picture painted mainstream audience forget charm forget romance ... oz n't mess around first episode ever saw struck nasty surreal could n't say ready watched developed taste oz got accustomed high level graphic violence not violence injustice crooked guard 'll sold nickel inmate 'll kill order get away well mannered middl

- RAW vs CLEANED

In [24]:
print("RAW REVIEW:\n")
print(sample_review)

print("\nCLEANED REVIEW:\n")
print(cleaned_review)

RAW REVIEW:

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is d

####  `Preprocessing function:`

##### Function Then applying to DataSet:

In [25]:
# prepare stopwords
stop_words = set(stopwords.words("english"))

stop_words.discard("not")
stop_words.discard("no")
stop_words.discard("nor")

# create a lemmatizer
lemmatizer = WordNetLemmatizer()


# The functions
def preprocess_text(text):
    
    # 1- tokenize
    tokens = word_tokenize(text)
    
    # 2- lowercase
    tokens = [token.lower()
              for token in tokens]
    
    # 3- remove punctuation
    tokens = [token 
              for token in tokens
              if token not in string.punctuation]
    
    # 4- remove stop words
    tokens = [token
              for token in tokens
              if token not in stop_words]
    
    # 5- lemmatize
    tokens = [lemmatizer.lemmatize(token)
              for token in tokens]
    
    # JOIN back into text 
    cleaned_text = " ".join(tokens)
    
    return cleaned_text
    

In [ ]:
# Applying to the full dataset
df["cleaned_review"] = df["review"].apply(preprocess_text)

In [28]:
df[ ["review", "cleaned_review", "sentiment"]].head()

,review,cleaned_review,sentiment
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching 1 oz episode '...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...,positive
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,positive
3,Basically there's a family where a little boy ...,basically 's family little boy jake think 's z...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei 's `` love time money '' visuall...,positive


##### compare few original vs cleaned reviews: 

In [29]:
for i in range(3):
    print(f"--- REVIEW {i+1} ---")

    print("\nRAW:")
    print(df["review"].iloc[i])

    print("\nCLEANED:")
    print(df["cleaned_review"].iloc[i])

    print("\n" + "=" * 80 + "\n")

--- REVIEW 1 ---

RAW:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the

#### Text Preprocessing Choices

The IMDb reviews were tokenized, converted to lowercase, cleaned of punctuation, filtered using English stop words, and lemmatized.

Stop-word removal reduces common low-information words and helps simplify the vocabulary.       
 However, the negation words not, no, and nor were intentionally preserved because they can change the meaning of a sentence in sentiment analysis.     

Lemmatization was used to reduce related word forms to a more consistent base form while preserving meaningful words.

The cleaned reviews were compared with the original reviews to verify that important sentiment information was not removed.